In [2]:
import pandas as pd
import lightgbm as lgb

train = pd.read_csv('../preprocessing/train.csv')
val   = pd.read_csv('../preprocessing/val.csv')

# X 컬럼 (date는 제외 — 분할용이었음)
X_cols = ['station_id', 'hour', 'dow', 'is_weekend', 'month', 'is_holiday',
          'temp', 'humid', 'wind', 'precip', 'pm10', 'pm25', 'ozone']

def prep(df):
    X = df[X_cols].copy()
    X['station_id'] = X['station_id'].astype('category')   # LightGBM에 범주형이라고 알려줌
    return X

X_train, X_val = prep(train), prep(val)
y_train_rent,   y_val_rent   = train['rent_cnt'],   val['rent_cnt']
y_train_return, y_val_return = train['return_cnt'], val['return_cnt']

print(f"train {X_train.shape} / val {X_val.shape}")

train (150024, 13) / val (14136, 13)


In [7]:
# early stopping: val 성능이 50라운드 동안 안 좋아지면 알아서 멈춤 (과적합 방지)
model_rent = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05,
                               objective='poisson',
                                random_state=42)

model_rent.fit(
    X_train, y_train_rent,
    eval_set=[(X_val, y_val_rent)],
    eval_metric='mae',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],  # 100라운드마다 로그
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002853 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1719
[LightGBM] [Info] Number of data points in the train set: 150024, number of used features: 13
[LightGBM] [Info] Start training from score -0.313885
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 0.519543	valid_0's poisson: 0.532884
[200]	valid_0's l1: 0.505609	valid_0's poisson: 0.517534
Early stopping, best iteration is:
[235]	valid_0's l1: 0.504448	valid_0's poisson: 0.516115


,learning_rate,0.05
,n_estimators,2000
,objective,'poisson'
,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001


In [8]:
model_return = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, 
                                 objective='poisson',
                                 random_state=42)

model_return.fit(
    X_train, y_train_return,
    eval_set=[(X_val, y_val_return)],
    eval_metric='mae',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002826 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1719
[LightGBM] [Info] Number of data points in the train set: 150024, number of used features: 13
[LightGBM] [Info] Start training from score -0.311234
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 0.53037	valid_0's poisson: 0.556577
[200]	valid_0's l1: 0.512989	valid_0's poisson: 0.537054
Early stopping, best iteration is:
[229]	valid_0's l1: 0.510937	valid_0's poisson: 0.53481


,learning_rate,0.05
,n_estimators,2000
,objective,'poisson'
,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001


In [13]:
from sklearn.metrics import mean_absolute_error

naive = y_train_rent.mean()

# 모델 출력(소수점) 그대로 평가
pred_rent   = model_rent.predict(X_val)
pred_return = model_return.predict(X_val)

mae_rent   = mean_absolute_error(y_val_rent,   pred_rent)
mae_return = mean_absolute_error(y_val_return, pred_return)
mae_naive  = mean_absolute_error(y_val_rent,   [naive]*len(y_val_rent))

print(f"대여 MAE: {mae_rent:.3f}  (평균으로 찍었을 때: {mae_naive:.3f})")
print(f"반납 MAE: {mae_return:.3f}")

대여 MAE: 0.504  (평균으로 찍었을 때: 0.777)
반납 MAE: 0.511


In [16]:
# val에 예측값 붙이기
val_check = val.copy()
val_check['pred_rent'] = model_rent.predict(X_val)
val_check['err'] = (val_check['pred_rent'] - val_check['rent_cnt']).abs()

# 1) 스테이션별 MAE (실제 수요 크기와 같이 보기)
by_st = val_check.groupby('station_id').agg(
    실제평균수요=('rent_cnt', 'mean'),
    MAE=('err', 'mean'),
).sort_values('실제평균수요', ascending=False).round(2)
print(by_st.to_string())

# 2) 시간대별 MAE
by_hr = val_check.groupby('hour').agg(
    실제평균수요=('rent_cnt', 'mean'),
    MAE=('err', 'mean'),
).round(2)
print(by_hr.to_string())

            실제평균수요   MAE
station_id              
ST0065        1.63  1.11
ST0685        1.02  0.98
ST1067        0.87  0.87
ST0627        0.45  0.69
ST0641        0.37  0.45
ST0107        0.37  0.55
ST0056        0.34  0.50
ST0720        0.31  0.41
ST0639        0.27  0.41
ST1302        0.18  1.04
ST1089        0.15  0.32
ST0642        0.14  0.33
ST0934        0.13  0.36
ST0342        0.13  0.30
ST1303        0.07  0.53
ST1330        0.07  0.24
ST0082        0.07  0.25
ST0628        0.06  0.16
ST1334        0.02  0.06
      실제평균수요   MAE
hour              
0       0.00  0.02
1       0.00  0.02
2       0.00  0.02
3       0.00  0.02
4       0.00  0.02
5       0.05  0.11
6       0.14  0.22
7       0.33  0.43
8       0.39  0.48
9       0.32  0.52
10      0.33  0.53
11      0.42  0.59
12      0.49  0.72
13      0.45  0.81
14      0.64  0.85
15      0.66  0.96
16      0.78  0.96
17      0.89  0.98
18      0.81  0.90
19      0.50  0.72
20      0.41  0.64
21      0.33  0.63
22      0.20  0.49


In [14]:
imp = pd.DataFrame({
    '피처': X_cols,
    '중요도': model_rent.feature_importances_,
}).sort_values('중요도', ascending=False)
print(imp.to_string(index=False))

        피처  중요도
station_id 1985
      hour 1697
      temp  795
     month  638
     humid  510
    precip  404
       dow  340
      pm10  180
     ozone  164
      wind  145
is_holiday   99
      pm25   93
is_weekend    0


In [17]:
import joblib
import json

# 모델 저장 (training 노트북 기준, 타슈관련/models 폴더로)
joblib.dump(model_rent,   '../models/model_rent_v1.pkl')
joblib.dump(model_return, '../models/model_return_v1.pkl')

# 모델 정보 메모 — 나중에 "이 모델 뭐였지?" 방지용
meta = {
    'version': 'v1',
    'trained_at': '2026-07-07',
    'dataset': 'tashu_ml_dataset_v1.csv',
    'train_period': '2025-01-01 ~ 2025-11-30',
    'model': 'LightGBM (objective=poisson, n_estimators=2000, lr=0.05, early_stopping=50)',
    'features': X_cols,                       # 서빙 때 입력 순서 맞추는 데 필요!
    'val_mae_rent': round(mae_rent, 3),       # 0.504
    'val_mae_return': round(mae_return, 3),   # 0.511
}
with open('../models/model_v1_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("저장 완료: model_rent_v1.pkl / model_return_v1.pkl / model_v1_meta.json")

저장 완료: model_rent_v1.pkl / model_return_v1.pkl / model_v1_meta.json


In [18]:
import joblib

model_rent   = joblib.load('../models/model_rent_v1.pkl')
model_return = joblib.load('../models/model_return_v1.pkl')

In [19]:
X_cols = ['station_id', 'hour', 'dow', 'is_weekend', 'month', 'is_holiday',
          'temp', 'humid', 'wind', 'precip', 'pm10', 'pm25', 'ozone']

def prep(df):
    X = df[X_cols].copy()
    X['station_id'] = X['station_id'].astype('category')
    return X

In [20]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# test 로드 & 준비 (학습 때와 동일한 전처리)
test = pd.read_csv('../preprocessing/test.csv')
X_test = prep(test)

def evaluate(model, y_true, name):
    pred = model.predict(X_test)
    mae  = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r2   = r2_score(y_true, pred)
    print(f"[{name}]  MAE: {mae:.3f}   RMSE: {rmse:.3f}   R²: {r2:.3f}")
    return pred

print(f"test 기간: {test['date'].min()} ~ {test['date'].max()}  ({len(test):,}행)\n")

pred_test_rent   = evaluate(model_rent,   test['rent_cnt'],   '대여')
pred_test_return = evaluate(model_return, test['return_cnt'], '반납')

# val과 비교 — 비슷하면 안정적이라는 뜻
print(f"\n참고) val에서는  대여 {mae_rent:.3f} / 반납 {mae_return:.3f}")

test 기간: 2026-02-01 ~ 2026-02-28  (12,768행)

[대여]  MAE: 0.461   RMSE: 0.843   R²: 0.485
[반납]  MAE: 0.469   RMSE: 0.817   R²: 0.487

참고) val에서는  대여 0.504 / 반납 0.511
